In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import linalg as la
from tqdm import tqdm
import pickle

import kwant

from alter_morph.kwant_utilities import (
    crack_hamiltonian_for_contacts_kwant,
    attach_leads_to_cracked,
)
from alter_morph.hamiltonians import alt_hamiltonian, spectral_function, _hopping_matrix
from alter_morph.mean_field import hartree_fock

matplotlib.rcParams.update(
    {
        "font.size": 11,
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": ["Computer Modern"],
    }
)

some functions:

In [16]:
def fermi_level(energies, filling):
    return np.mean(
        energies[int(filling * len(energies)) - 1 : int(filling * len(energies)) + 1]
    )


def dos_at_fermi(energies, fermi_energy, broadening):
    norm_lorenzian = (1 / np.pi) * (
        broadening / ((energies - fermi_energy) ** 2 + broadening**2)
    )
    return np.mean(norm_lorenzian)

In [17]:
data_name = "data/full_phase_diagram_20_n_V1"

# load data
with open(data_name + ".pickle", "rb") as f:
    data = pickle.load(f)

# load lattice
with open("analysis/voronoi_20.pickle", "rb") as f:
    lattice = pickle.load(f)

In [18]:
# process the data

fillings = data["filling"]
J_vals = data["J"]

all_avg_mags = np.zeros([len(fillings), len(J_vals)])
all_energies = np.zeros([len(fillings), len(J_vals), lattice.n_vertices * 4])

broadening = 0.01
all_dos_vals = np.zeros([len(fillings), len(J_vals)])

all_bond_diff_m = np.zeros([len(fillings), len(J_vals)])
all_bond_diff_n = np.zeros([len(fillings), len(J_vals)])

for i, filling in enumerate(tqdm(fillings)):
    for j, J in enumerate(J_vals):
        current_data = data["params"][i, j]

        # calculate the average magnetization
        all_avg_mags[i, j] = np.mean(current_data["m"])

        # calculate the energies
        ham = alt_hamiltonian(
            lattice,
            current_data["t1"],
            current_data["t2"],
            current_data["J"],
            current_data["U"],
            current_data["m"],
            current_data["n"],
        )
        all_energies[i, j] = la.eigvalsh(ham)

        # calculate the density of states
        all_dos_vals[i, j] = dos_at_fermi(
            all_energies[i, j], fermi_level(all_energies[i, j], filling), broadening
        )

        # difference in m on bonds
        m = data["params"][i, j]["m"]
        m_on_bonds = m[lattice.edges.indices]
        all_bond_diff_m[i, j] = np.abs(m_on_bonds[:, 0] - m_on_bonds[:, 1]).mean()

        # difference in n on bonds
        n = data["params"][i, j]["n"]
        n_on_bonds = n[lattice.edges.indices]
        all_bond_diff_n[i, j] = np.abs(n_on_bonds[:, 0] - n_on_bonds[:, 1]).mean()

# save the processed data
with open(data_name + "_processed.pickle", "wb") as f:
    pickle.dump(
        {
            "fillings": fillings,
            "J_vals": J_vals,
            "all_avg_mags": all_avg_mags,
            "all_energies": all_energies,
            "all_dos_vals": all_dos_vals,
            "all_bond_diff_m": all_bond_diff_m,
            "all_bond_diff_n": all_bond_diff_n,
        },
        f,
    )

  0%|          | 0/21 [00:00<?, ?it/s]

100%|██████████| 21/21 [09:30<00:00, 27.19s/it]
